# Assignment 1

![](https://media.giphy.com/media/xT9C25UNTwfZuk85WP/giphy-downsized-large.gif)

Remember the rules of ~Fight~ Code Club:
1. ALWAYS DOCUMENT
2. Cite resources that you use (paste links)
3. Include the names people who you worked with
4. Be neat and organized

## Scrape and Clean data

Based on you proposal, scrape or collect your data:

1. One variable must be either: (40 pts)
    1. scraped from the web OR;
    2. collected from an API AND you must create one new variable that is "new" to the best of your knowledge (combination of other variables representing something new).

2. You must have at least 3 variables, but you may include as many as you want into you final dataset. Likely, you will want to include more to make graphs and regressions. (30pts)

3. You must *be able* to run a regression that makes some sense with this data (the regression doesn't have to be a complete model). Briefly describe one regression you would run with your variables. (**DO NOT** run a regression, yet). (15pts)

4. You must have one combined and cleaned dataset (15 pts)

You must submit one python notebook on how you scraped/gathered data from an api, and how you combined and cleaned you data. I should be able to run your code and reproduce your final data set.  

The other variables that you choose to include do not have to be collected by API or webscraped, but you do have to combine the files and clean the dataset with python.

Thus, you must submit:
- Your finalized data set (only one) (note: you may add more variables in the future).
- Your documented python notebook
- Any associated data files needed to produce the final dataset.

You will be evaluated on:
- Completeness of the data
- Quality of the code 
- The creativity of the new variable/webscraped data you gather

Be sure to upload ALL associated files for your code to run. I will run your code from beginning to end - make it easy for me to replicate your code.

# Portion 1
### Getting my Data...ultimately into a CSV


#### Setting up the tag getter

In [2]:
url = "http://ws.audioscrobbler.com/2.0/" #Url base for the API, used in my future F strings as inputs, F strings are awesome
YOUR_API_KEY = '915b23d080ae463f6850ac8cec3aefb6' #my API key,
artist= 'Cher'#cher is just a placeholder to make sure everything works correctly
import requests
import pandas as pd

toptagsreq=requests.get(f'{url}?method=artist.gettoptags&artist={artist}&api_key={YOUR_API_KEY}&format=json')# gets the top tag for an artist specified in the F STRING {ARTIST} blob, if i change the artist in the line at the beginning it will change from kanye
toptagget=toptagsreq.json() #returns our JSON for the request of the artist tags
toptagget # prints json for all of the tags of a given artist, cher's top tag in this case is pop

{'weeklychartlist': {'chart': [{'#text': '',
    'from': '1108296000',
    'to': '1108900800'},
   {'#text': '', 'from': '1108900800', 'to': '1109505600'},
   {'#text': '', 'from': '1109505600', 'to': '1110110400'},
   {'#text': '', 'from': '1110110400', 'to': '1110715200'},
   {'#text': '', 'from': '1110715200', 'to': '1111320000'},
   {'#text': '', 'from': '1111320000', 'to': '1111924800'},
   {'#text': '', 'from': '1111924800', 'to': '1112529600'},
   {'#text': '', 'from': '1112529600', 'to': '1113134400'},
   {'#text': '', 'from': '1113134400', 'to': '1113739200'},
   {'#text': '', 'from': '1113739200', 'to': '1114344000'},
   {'#text': '', 'from': '1114344000', 'to': '1114948800'},
   {'#text': '', 'from': '1114948800', 'to': '1115553600'},
   {'#text': '', 'from': '1115553600', 'to': '1116158400'},
   {'#text': '', 'from': '1116158400', 'to': '1116763200'},
   {'#text': '', 'from': '1116763200', 'to': '1117368000'},
   {'#text': '', 'from': '1117368000', 'to': '1117972800'},
   {

#### Getting number of weeks

In [ ]:
chartrequest=requests.get(f'{url}?method=tag.getweeklychartlist&tag=rock&api_key={YOUR_API_KEY}&format=json') 
charts=chartrequest.json()
charts
#whole thing lets me see all of the weeks that are counted for in the API, same structure as the cher tag request just a different endpoint
#includes rock as a tag but its not important, just a leftover code that doesn't really change anything
#this whole for loop lets me see how many weeks there are, roughly 21 years or 1105 weeks
#chartnum=0
#for i in charts['weeklychartlist']['chart']:
    #print(i,'week',chartnum)
    #chartnum+=1

#### Making my list

In [ ]:
chartnum=0
listylist=[] #creates an empty list for the artists to go in later
for i in charts['weeklychartlist']['chart']:
    fromloop = charts['weeklychartlist']['chart'][chartnum]['from'] #sets beginning of date search
    toloop = charts['weeklychartlist']['chart'][chartnum]['to'] #sets end of date search
    weeklisteningdata = requests.get(f'{url}?method=user.getweeklyartistchart&user=rj&api_key={YOUR_API_KEY}&format=json&to={toloop}&from={fromloop}')
    rjdata=weeklisteningdata.json()
    for j in range(0, min(250, len(rjdata['weeklyartistchart']['artist']))):
        listylist.append(rjdata['weeklyartistchart']['artist'][j]['name'])
        print(rjdata['weeklyartistchart']['artist'][j]['name']) #this line isn't necessary, just lets me see the artists I'm dealing with and how many
    chartnum+=1 #advances the chart number counter 1 item

listylist #prints my list of all of the total artists for each of the weeks, ends up being over 3500 artists after removing all duplicates 
#probably would want to expand this in another project

#### Making the dictionary, this takes a long time to run the whole thing so the CSV file also works after this

In [3]:
import urllib.parse as up # THIS IS TOTALLY STACK OVERFLOW I AM NOT GONNA ACT LIKE I KNEW HOW TO DO THIS
    #https://mojoauth.com/binary-encoding-decoding/percent-encoding-url-encoding-with-python#encoding-query-parameters
    #https://www.reddit.com/r/webdev/comments/2c5a6h/im_using_an_api_to_create_qr_codes_i_need_to_put/
import time

taglist = []
sickdict={}
counter=0
for item in listylist:
    itemencode = up.quote(item) #beautiful stack overflow help, pencode encodes the name so we can 
    if item not in sickdict: #checking to see if the artist we found is in the dictionary called sickdict
        toptagsreq=requests.get(f'{url}?method=artist.gettoptags&artist={itemencode}&api_key={YOUR_API_KEY}&format=json')# gets the top tag for an artist specified in the F STRING {itemencode} blob
        toptagget=toptagsreq.json()
        tags = toptagget['toptags']['tag'] #gets all of the tags for a given artist
        tag = tags[0]['name'] if len(tags) > 0 else 'unknown' # returns the top tag for that given artist, returns unknown if it doesnt know
        sickdict[item]= tag # puts item in the dictionary with the tag variable 
        print(item,tag,counter) # so i can see what is going on in the back end, again not a necessary line of code
    else: #these lines make it so that i'm just moving on in the case that item is already in the dictionary, doesnt waste time
        pass
    counter+=1 # lets me see the number of artist im on
    time.sleep(0.3) # THIS IS 100% a suggestion of AI I will not take credit, this is just so that the API doesn't take too many requests and explode and give me a JSONDecodeError,
sickdict #shows me the dictionary at the end of everything, this whole line took like an hour to run to 3500 artists I believe

NameError: name 'listylist' is not defined

In [ ]:

othercount=0
for artist in sickdict:
    looptag= sickdict[artist].lower()
    if 'rock' in looptag or 'grunge' in looptag or 'punk' in looptag or 'ska' in looptag or 'alternative' in looptag or 'britpop' in looptag:
        sickdict[artist]='Rock'
    elif 'pop' in looptag:
        sickdict[artist]='Pop'
    elif 'metal' in looptag:
        sickdict[artist]='Metal'
    elif 'jazz' in looptag or 'swing' in looptag or 'blues' in looptag or 'soul' in looptag or 'funk' in looptag:
        sickdict[artist]='Jazz'
    elif 'indie' in looptag:
        sickdict[artist]='Indie'
    elif 'electronic' in looptag or 'techno' in looptag or 'house' in looptag or 'ambient' in looptag or 'trance' in looptag or 'dubstep' in looptag or 'trip-hop' in looptag:
        sickdict[artist]='Electronic'
    elif 'soul' in looptag:
        sickdict[artist]='Soul'
    elif 'classical' in looptag:
        sickdict[artist]='Classical'
    elif 'funk' in looptag:
        sickdict[artist]='Funk'
    elif 'hip hop' in looptag or 'hip' in looptag or 'rap' in looptag or 'hop' in looptag:
        sickdict[artist]='Hip Hop'
    elif 'folk' in looptag or 'country' in looptag or 'americana' in looptag:
        sickdict[artist] = 'folk'
    else:
        sickdict[artist]='other'

sickdict

##### Next block is not necessary, just to see composition of sickdict

In [ ]:
import collections # https://www.reddit.com/r/learnpython/comments/ybmic0/count_unique_values_in_dictionary/
    #collections helped me see my different count of values for each of the genres
new_dict = collections.Counter(sickdict.values())
print(new_dict)
    #not super important, I just like to get a better feel for the data I'm working with, ultimately a lot of smaller artists either don't
    #have tags, or have tags that don't really classify the genre, something like 'seen live' or 'awesome' which kind of doesnt help me,
    #that's why theres so many others, this would be an expansion as well for a future project.

##### The actual bread and butter pretty much


In [ ]:
dataframe=[] #new 
chartnumber=0
url = "http://ws.audioscrobbler.com/2.0/"
YOUR_API_KEY = '915b23d080ae463f6850ac8cec3aefb6'
#Essentially my entire goal is to go through each week, find the total share of each genre during the week,

for chart in charts['weeklychartlist']['chart']:
    fromdate = charts['weeklychartlist']['chart'][chartnumber]['from'] #sets date to begin at for the weeklisteningdata search
    todate = charts['weeklychartlist']['chart'][chartnumber]['to']# sets end date for the weeklisteningdata
    weeklisteningdata = requests.get(f'{url}?method=user.getweeklyartistchart&user=rj&api_key={YOUR_API_KEY}&format=json&to={todate}&from={fromdate}')
   #the above line gets the weekly artist chart for the week specified by the fromdate and todate variables
    weekjson=weeklisteningdata.json() #gets the above request as a JSON print
    weekartists=weekjson['weeklyartistchart']['artist'][:250] #gets first 250 artists, representative of most of the listening for the week
    #print(weekartists,'_____________________________________', f'week{chartnumber}') #prints each week's top 250 with playcount and rank
    
    genreplays={} #creates an empty dictionary for every week in the loop, use this to see genre shares for each of the weeks
    totalplays=0 #initializes the total plays so we can see out of the total plays per week, how many were each genre
    for musician in weekartists: # cycles through each of the top 250 artists per week
        name=musician['name'] #gets artist's name
        plays=int(musician['playcount']) #how many times they were played during this week
        tag=sickdict.get(name,'other') #gets the name of the artist from the sick dictionary
        if tag not in genreplays: #without this line, the next line breaks my code, 
            genreplays[tag]=0 #including this to initialize each of the genres within the dict
        genreplays[tag]=genreplays.get(tag) + plays #silly refresher but i forgot how to do this https://www.digitalocean.com/community/tutorials/python-add-to-dictionary
        totalplays+=plays #adds the # of plays to total plays, measures how much the listener is listening per week
        print(musician['name'],tag, f'week{chartnumber}') #prints the musician, their tag, and the chart week number

    for genre, plays in genreplays.items(): #puts all of this into a list that I can get into a dataframe,
        dataframe.append({'Week':fromdate,'Genre':genre, 'Play Share': plays/totalplays,'Total Plays':totalplays}) 
        #fromdate is in a disgusting format, gonna look this up to change it, https://stackoverflow.com/questions/17594298/date-time-formats-in-python
    chartnumber+=1
    time.sleep(0.3)
df = pd.DataFrame(dataframe)
df

In [ ]:
df['Week']=pd.to_datetime(df['Week'], unit='ns') #convert to date
df=df.drop(columns=['date'])
df

#### This is the CSV File of my dataframe for artists


In [5]:
df.to_csv('UglyDF.csv') #PUT THIS DATA FRAME IN A CSV SO I CAN STOP REDOING THE ENTIRE THING

NameError: name 'df' is not defined

# SHORTCUT TO NOT RUN THE CODE FOR 2 HOURS
#### Getting the CSV (Should be attached in the folder, Name: UglyDF.csv)

In [23]:
path='/Users/alex/Documents/Pace/Classes/Junior/Spring/Data Analytics/Project/' #my path, I know it will be different
data=pd.read_csv(f'{path}UglyDF.csv') #brings the data Frame into the notebook from the path
data.columns

Index(['Unnamed: 0', 'Week', 'Genre', 'Play Share', 'Total Plays'], dtype='object')

In [22]:
# changing the dates to be a legible week instead of just gibberish 
#https://stackoverflow.com/questions/19645706/last-fm-api-date-range  God i love stack overflow
data['Month']=pd.to_datetime(data['Week'], unit='ns').dt.month
data['Year']=pd.to_datetime(data['Week'], unit='ns').dt.year
data['Play Share (Number)']=data['Play Share'] * data['Total Plays']
data

,Unnamed: 0,Week,Genre,Play Share,Total Plays,Month,Year,Play Share (Number)
0,0,2005-02-13,Rock,0.516038,17084,2,2005,8816.0
1,1,2005-02-13,Metal,0.074631,17084,2,2005,1275.0
2,2,2005-02-13,Jazz,0.133868,17084,2,2005,2287.0
3,3,2005-02-13,Electronic,0.058125,17084,2,2005,993.0
4,4,2005-02-13,other,0.128424,17084,2,2005,2194.0
...,...,...,...,...,...,...,...,...
5111,5111,2026-04-12,Pop,0.032258,31,4,2026,1.0
5112,5112,2026-04-19,Pop,0.474576,59,4,2026,28.0
5113,5113,2026-04-19,Rock,0.406780,59,4,2026,24.0
5114,5114,2026-04-19,other,0.101695,59,4,2026,6.0


#### Fred data now

In [12]:
# pip install fredapi # this entire thing was shown by this https://pypi.org/project/fredapi/ 
from fredapi import Fred # gets Fred Package
FREDKEY= '2432515d4bc3acaf9cc6bdbbc8ad8558' #gets my fred API key
fred=Fred(api_key=FREDKEY) #structures the variable to use my API key when getting data
recession=fred.get_series('USREC') #recession dummy variable
PCEPI=fred.get_series('PCEPI') # PCE variable
UNEMP=fred.get_series('UNRATE') #U3 Unemployment rate
UMICH=fred.get_series('UMCSENT') #UMICH Consumer sentiment Survey

#### Fred Data Frame

In [15]:
freddf=pd.DataFrame(PCEPI)
freddf['REC']=pd.DataFrame(recession) #MAKES A COLUMN FOR EACH VARIABLE
freddf['UNRATE']=pd.DataFrame(UNEMP)
freddf['PCE']=pd.DataFrame(PCEPI)
freddf['UMICH']=pd.DataFrame(UMICH)
freddf.drop(freddf.columns[0],axis=1) #Drops the first initial useless column
freddf['Year']=freddf.index #Makes a column from the year, given by the index but just so i can match it with freds year data
freddf['Month']=pd.to_datetime(freddf['Year'], unit='ns').dt.month
freddf['Year']=pd.to_datetime(freddf['Year'], unit='ns').dt.year
 #https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html#:~:text=*%20pandas.Series.ffill.%20*%20pandas.Series.fillna.%20*%20pandas.Series.interpolate.
freddf

,0,REC,UNRATE,PCE,UMICH,Year,Month
1959-01-01,15.164,0.0,6.0,15.164,NaN,1959,1
1959-02-01,15.179,0.0,5.9,15.179,NaN,1959,2
1959-03-01,15.189,0.0,5.6,15.189,NaN,1959,3
1959-04-01,15.219,0.0,5.2,15.219,NaN,1959,4
1959-05-01,15.227,0.0,5.1,15.227,95.3,1959,5
...,...,...,...,...,...,...,...
2025-10-01,127.871,0.0,NaN,127.871,53.6,2025,10
2025-11-01,128.152,0.0,4.5,128.152,51.0,2025,11
2025-12-01,128.576,0.0,4.4,128.576,52.9,2025,12
2026-01-01,128.965,0.0,4.3,128.965,56.4,2026,1


In [16]:
finaldf=pd.merge(data,freddf, on=['Year','Month'], how='left')
finaldf #Merges both of the dataframes into one

,Unnamed: 0,Week,Genre,Play Share,Total Plays,Month,Year,Play Share (Number),0,REC,UNRATE,PCE,UMICH
0,0,2005-02-13,Rock,0.516038,17084,2,2005,8816.0,81.132,0.0,5.4,81.132,94.1
1,1,2005-02-13,Metal,0.074631,17084,2,2005,1275.0,81.132,0.0,5.4,81.132,94.1
2,2,2005-02-13,Jazz,0.133868,17084,2,2005,2287.0,81.132,0.0,5.4,81.132,94.1
3,3,2005-02-13,Electronic,0.058125,17084,2,2005,993.0,81.132,0.0,5.4,81.132,94.1
4,4,2005-02-13,other,0.128424,17084,2,2005,2194.0,81.132,0.0,5.4,81.132,94.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5111,5111,2026-04-12,Pop,0.032258,31,4,2026,1.0,NaN,NaN,NaN,NaN,NaN
5112,5112,2026-04-19,Pop,0.474576,59,4,2026,28.0,NaN,NaN,NaN,NaN,NaN
5113,5113,2026-04-19,Rock,0.406780,59,4,2026,24.0,NaN,NaN,NaN,NaN,NaN
5114,5114,2026-04-19,other,0.101695,59,4,2026,6.0,NaN,NaN,NaN,NaN,NaN


In [17]:
finaldf = finaldf[finaldf['Year'] >= 2005] #Excludes FRED data before 2005, the Lastfm api only goes until 2005
finaldf.drop(columns=0) #drops an empty column
finaldf = finaldf[finaldf['Year'] < 2026] # Excludes some data to be at 20 years
finaldf

,Unnamed: 0,Week,Genre,Play Share,Total Plays,Month,Year,Play Share (Number),0,REC,UNRATE,PCE,UMICH
0,0,2005-02-13,Rock,0.516038,17084,2,2005,8816.0,81.132,0.0,5.4,81.132,94.1
1,1,2005-02-13,Metal,0.074631,17084,2,2005,1275.0,81.132,0.0,5.4,81.132,94.1
2,2,2005-02-13,Jazz,0.133868,17084,2,2005,2287.0,81.132,0.0,5.4,81.132,94.1
3,3,2005-02-13,Electronic,0.058125,17084,2,2005,993.0,81.132,0.0,5.4,81.132,94.1
4,4,2005-02-13,other,0.128424,17084,2,2005,2194.0,81.132,0.0,5.4,81.132,94.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5043,5043,2025-12-14,Pop,0.042857,70,12,2025,3.0,128.576,0.0,4.4,128.576,52.9
5044,5044,2025-12-14,Rock,0.028571,70,12,2025,2.0,128.576,0.0,4.4,128.576,52.9
5045,5045,2025-12-21,other,0.950000,60,12,2025,57.0,128.576,0.0,4.4,128.576,52.9
5046,5046,2025-12-21,Pop,0.050000,60,12,2025,3.0,128.576,0.0,4.4,128.576,52.9


In [19]:
groupedfinal=finaldf[['Genre','Play Share','Play Share (Number)','Total Plays','REC', 'UNRATE','PCE','UMICH','Year','Month','Week']].reset_index().groupby(['Week','Genre']).sum()
groupedfinal.to_csv('RealFinalDF.csv')

# Part 2: Questions

##### I will be doing regressions to see how changes in the recession dummy variable, total plays, unemployment rate and the UMICH survey relate to changes in play share for each genre. playshare= b0 + b1*rec + b2*unrate+ b3*totalplays + e
##### I might also do changes in total plays based on the unemployment rate and UMICH Survey, maybe I can get more macro data with sentiment. totalplays=  b0+b1*rec+b2*unrate+b3*(other macro variable) + e